Steps involved

1. Define State
2. Create LLM
3. Create Chat Node
4. Build Graph(node+edge)
5. Compile
6. Test single Run
7. Add loop for real chat 

In [11]:
from dotenv import load_dotenv
load_dotenv()

import os

In [12]:
hf_token=os.getenv("HF_TOKEN")

# Add a checkpointer

In [13]:
from langgraph.checkpoint.memory import InMemorySaver

memory=InMemorySaver()

In [14]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

# Define State

In [15]:
class State(TypedDict):
    messages:Annotated[list,add_messages]
    

# Create LLM


In [16]:
llm=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="conversational",
    huggingfacehub_api_token=hf_token,
    max_new_tokens=200,
    temperature=0.7
)

llm = ChatHuggingFace(llm=llm)

# Chat node

In [21]:
def chatbot(state:State):
    #convert LangGraph messages-> plain prompt
    response=llm.invoke(state['messages'])
    return {"messages":[response]}

# Build and Compile Graph

In [22]:
graph=StateGraph(State)

graph.add_node("chatbot",chatbot)


graph.add_edge(START,"chatbot")
graph.add_edge("chatbot",END)
#change will be here
app=graph.compile(checkpointer=memory)

# Define Config

In [23]:
config={
    "configurable":{
        "thread_id":"user_1"
    }
}

# Test single run 

In [24]:
#initial empty state

test_state={
    "messages":[HumanMessage(content="Say hello in one sentence")]
}

#run graph once
result=app.invoke(test_state,config=config)

#print response
print("Bot", result["messages"][-1].content)
    

Bot Hello there!


# Add loop for real chat

In [27]:
#now no need to maintain state

print("Persistance Chat bot is ready! Type 'exit' to stop \n")

while True:
    user_input=input("You: ")

    if(user_input.lower())=="exit":
        print("Bye Bye ")
        break

    result=app.invoke(
        {"messages":[HumanMessage(content=user_input)]},
        config=config
    )

    print("Bot:",result["messages"][-1].content)

    

Persistance Chat bot is ready! Type 'exit' to stop 



You:  What is my Name?


Bot: Your name is Samart.


You:  I love Ai ml


Bot: Great! It's wonderful to hear that you love AI and ML. How can I assist you with your interests in AI and ML?


You:  what does i love?


Bot: In your previous statement, "I love AI ML," you expressed that you love AI (Artificial Intelligence) and ML (Machine Learning). If you have a specific question or need more information about AI and ML, feel free to share!


You:  exit


Bye Bye 


In [26]:
snapshot=app.get_state(config)

print(snapshot)

StateSnapshot(values={'messages': [HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='5eb02125-0c32-42c5-82af-098d14e971ac'), HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='81d3341b-fa1f-41c9-b640-99e056848831'), AIMessage(content='Hello there!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 44, 'total_tokens': 48}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0f1a-6c4d-7e52-8c87-2a638f247bdf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 4, 'total_tokens': 48}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='010d6cde-a926-4fa4-a9f9-b7daef50360d'), AIMessage(content='Hi there! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens

In [28]:
snapshot=app.get_state(config)

print(snapshot)

StateSnapshot(values={'messages': [HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='5eb02125-0c32-42c5-82af-098d14e971ac'), HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='81d3341b-fa1f-41c9-b640-99e056848831'), AIMessage(content='Hello there!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 44, 'total_tokens': 48}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0f1a-6c4d-7e52-8c87-2a638f247bdf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 4, 'total_tokens': 48}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='010d6cde-a926-4fa4-a9f9-b7daef50360d'), AIMessage(content='Hi there! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens

In [29]:
print(snapshot.values['messages'])

[HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='5eb02125-0c32-42c5-82af-098d14e971ac'), HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='81d3341b-fa1f-41c9-b640-99e056848831'), AIMessage(content='Hello there!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 44, 'total_tokens': 48}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0f1a-6c4d-7e52-8c87-2a638f247bdf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 4, 'total_tokens': 48}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='010d6cde-a926-4fa4-a9f9-b7daef50360d'), AIMessage(content='Hi there! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 58, 'total

In [30]:
# view all checkpoints

In [32]:
history=app.get_state_history(config)

for checkpoint in history:
    print(checkpoint)
    print("-"*50)

StateSnapshot(values={'messages': [HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='5eb02125-0c32-42c5-82af-098d14e971ac'), HumanMessage(content='Say hello in one sentence', additional_kwargs={}, response_metadata={}, id='81d3341b-fa1f-41c9-b640-99e056848831'), AIMessage(content='Hello there!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 44, 'total_tokens': 48}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0f1a-6c4d-7e52-8c87-2a638f247bdf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 4, 'total_tokens': 48}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='010d6cde-a926-4fa4-a9f9-b7daef50360d'), AIMessage(content='Hi there! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens